# Phase 7: Final Ensemble — Multi-Seed + Rank Averaging

**Author:** Taline Zeidan  
**Course:** COE 546 — Machine Learning, Spring 2026  
**Input:** `data/train_features.csv` (V1), `data/train_features_v2.csv` (V2)  
**Output:** `outputs/submission_final.csv`

---

## Strategy

From our experiments so far:
- **XGBoost V1** (29 features) = best single model, public LB **0.97031**
- **LightGBM V2** (52 features) = V2 features helped LightGBM (+0.00037 CV AUC)
- **CatBoost V1** (29 features) = strong diverse model
- Adding more features to XGBoost **hurt** it — V1 features are optimal for XGB

This notebook:
1. Reruns XGBoost V1 with **5 different seeds** — averages reduce variance
2. Reruns LightGBM V2 with best params
3. Reruns CatBoost V1 with best params
4. Tries **rank averaging** vs probability averaging vs stacking
5. Submits the best

**Why multi-seed XGBoost?**  
Each seed produces slightly different trees due to subsampling randomness. Averaging 5 seeds reduces prediction variance without changing the model — essentially free performance gain.

**Why rank averaging?**  
Converting predictions to ranks before averaging removes the effect of different probability scales across models. It is more robust than raw probability averaging when models have different calibrations.

## 1. Imports & Setup

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ── Paths ───────────────────────────────────────────────────────────────────
BASE_DIR    = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR    = os.path.join(BASE_DIR, 'data')
OUT_DIR     = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

TRAIN_V1    = os.path.join(DATA_DIR, 'train_features.csv')
TEST_V1     = os.path.join(DATA_DIR, 'test_features.csv')
TRAIN_V2    = os.path.join(DATA_DIR, 'train_features_v2.csv')
TEST_V2     = os.path.join(DATA_DIR, 'test_features_v2.csv')
SUB_PATH    = os.path.join(OUT_DIR,  'submission_final.csv')

# ── Constants ───────────────────────────────────────────────────────────────
SEEDS            = [42, 123, 456, 789, 2026]  # 5 seeds for multi-seed averaging
N_FOLDS          = 5
SCALE_POS_WEIGHT = 33.4
EARLY_STOPPING   = 50

print('Imports done.')
print(f'Seeds: {SEEDS}')

Imports done.
Seeds: [42, 123, 456, 789, 2026]


## 2. Load Data

In [2]:
# ── V1 features (29) — best for XGBoost and CatBoost ──────────────────────
train_v1 = pd.read_csv(TRAIN_V1)
test_v1  = pd.read_csv(TEST_V1)
FEAT_V1  = [c for c in train_v1.columns if c not in ['id', 'order_placed']]
X_v1     = train_v1[FEAT_V1]
y        = train_v1['order_placed']
Xt_v1    = test_v1[FEAT_V1]

# ── V2 features (52) — best for LightGBM ──────────────────────────────────
train_v2 = pd.read_csv(TRAIN_V2)
test_v2  = pd.read_csv(TEST_V2)
FEAT_V2  = [c for c in train_v2.columns if c not in ['id', 'order_placed']]
X_v2     = train_v2[FEAT_V2]
Xt_v2    = test_v2[FEAT_V2]

print(f'V1 features: {len(FEAT_V1)} | Train: {X_v1.shape} | Test: {Xt_v1.shape}')
print(f'V2 features: {len(FEAT_V2)} | Train: {X_v2.shape} | Test: {Xt_v2.shape}')
print(f'Target: {y.sum():,} positives ({y.mean()*100:.2f}%)')

V1 features: 29 | Train: (297236, 29) | Test: (99639, 29)
V2 features: 52 | Train: (297236, 52) | Test: (99639, 52)
Target: 8,644 positives (2.91%)


## 3. CV Helper & Rank Averaging Utility

In [3]:
def run_cv(model_fn, X, y, X_test, n_folds=N_FOLDS, seed=42, label='Model'):
    skf        = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds  = np.zeros(len(X))
    test_preds = np.zeros(len(X_test))
    fold_aucs  = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        oof_fold, test_fold    = model_fn(X_tr, y_tr, X_val, y_val)
        oof_preds[val_idx]     = oof_fold
        test_preds            += test_fold / n_folds
        fold_auc = roc_auc_score(y_val, oof_fold)
        fold_aucs.append(fold_auc)
        print(f'  [{label}] Fold {fold}/{n_folds} — AUC: {fold_auc:.5f}')
    mean_auc = np.mean(fold_aucs)
    oof_auc  = roc_auc_score(y, oof_preds)
    print(f'  [{label}] OOF AUC: {oof_auc:.5f} | Mean: {mean_auc:.5f} ± {np.std(fold_aucs):.5f}')
    return oof_preds, test_preds, oof_auc

def rank_avg(preds_list):
    """Convert each prediction array to ranks, then average."""
    ranked = [rankdata(p) / len(p) for p in preds_list]
    return np.mean(ranked, axis=0)

print('Helpers defined.')

Helpers defined.


## 4. Multi-Seed XGBoost V1

We run XGBoost V1's best params with 5 different random seeds. Each seed produces slightly different trees due to subsampling randomness. Averaging 5 seeds is equivalent to training an ensemble of 5 models — it reduces variance without any hyperparameter changes.

In [4]:
# Best XGBoost V1 params (from 04_modeling.ipynb Optuna run)
XGB_BEST_PARAMS = {
    'n_estimators'     : 1500,
    'learning_rate'    : 0.01381105528962201,
    'max_depth'        : 9,
    'min_child_weight' : 46,
    'subsample'        : 0.9280864631542471,
    'colsample_bytree' : 0.68577731284726,
    'reg_alpha'        : 2.75787843275108,
    'reg_lambda'       : 3.98468850175631,
    'scale_pos_weight' : SCALE_POS_WEIGHT,
    'eval_metric'      : 'auc',
    'use_label_encoder': False,
    'n_jobs'           : -1,
    'verbosity'        : 0,
}

all_oof_xgb  = []
all_test_xgb = []
all_auc_xgb  = []

for seed in SEEDS:
    print(f'\nRunning XGBoost seed={seed}...')
    params = {**XGB_BEST_PARAMS, 'random_state': seed}

    def xgb_fn(X_tr, y_tr, X_val, y_val):
        model = xgb.XGBClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set              = [(X_val, y_val)],
            early_stopping_rounds = EARLY_STOPPING,
            verbose               = False
        )
        return model.predict_proba(X_val)[:, 1], model.predict_proba(Xt_v1)[:, 1]

    oof, test, auc = run_cv(xgb_fn, X_v1, y, Xt_v1, seed=seed, label=f'XGB-s{seed}')
    all_oof_xgb.append(oof)
    all_test_xgb.append(test)
    all_auc_xgb.append(auc)

# Average across seeds
oof_xgb_avg  = np.mean(all_oof_xgb,  axis=0)
test_xgb_avg = np.mean(all_test_xgb, axis=0)
auc_xgb_avg  = roc_auc_score(y, oof_xgb_avg)

# Rank average across seeds
oof_xgb_rank  = rank_avg(all_oof_xgb)
test_xgb_rank = rank_avg(all_test_xgb)
auc_xgb_rank  = roc_auc_score(y, oof_xgb_rank)

print(f'\n✅ Multi-seed XGBoost Results:')
print(f'   Individual seed AUCs : {[round(a,5) for a in all_auc_xgb]}')
print(f'   Mean of individuals  : {np.mean(all_auc_xgb):.5f}')
print(f'   Prob avg OOF AUC     : {auc_xgb_avg:.5f}')
print(f'   Rank avg OOF AUC     : {auc_xgb_rank:.5f}')
print(f'   Single seed (V1 LB)  : 0.97031')


Running XGBoost seed=42...
  [XGB-s42] Fold 1/5 — AUC: 0.96998
  [XGB-s42] Fold 2/5 — AUC: 0.97069
  [XGB-s42] Fold 3/5 — AUC: 0.96919
  [XGB-s42] Fold 4/5 — AUC: 0.97104
  [XGB-s42] Fold 5/5 — AUC: 0.97177
  [XGB-s42] OOF AUC: 0.97045 | Mean: 0.97053 ± 0.00089

Running XGBoost seed=123...
  [XGB-s123] Fold 1/5 — AUC: 0.97072
  [XGB-s123] Fold 2/5 — AUC: 0.97149
  [XGB-s123] Fold 3/5 — AUC: 0.97034
  [XGB-s123] Fold 4/5 — AUC: 0.97075
  [XGB-s123] Fold 5/5 — AUC: 0.97078
  [XGB-s123] OOF AUC: 0.97075 | Mean: 0.97082 ± 0.00037

Running XGBoost seed=456...
  [XGB-s456] Fold 1/5 — AUC: 0.97190
  [XGB-s456] Fold 2/5 — AUC: 0.96865
  [XGB-s456] Fold 3/5 — AUC: 0.97224
  [XGB-s456] Fold 4/5 — AUC: 0.97106
  [XGB-s456] Fold 5/5 — AUC: 0.97008
  [XGB-s456] OOF AUC: 0.97075 | Mean: 0.97079 ± 0.00130

Running XGBoost seed=789...
  [XGB-s789] Fold 1/5 — AUC: 0.96963
  [XGB-s789] Fold 2/5 — AUC: 0.97142
  [XGB-s789] Fold 3/5 — AUC: 0.97116
  [XGB-s789] Fold 4/5 — AUC: 0.97075
  [XGB-s789] Fold 5/

## 5. LightGBM V2 (Best Features for LGB)

In [5]:
# Best LightGBM V2 params (from 06_modeling_v2.ipynb Optuna run)
LGB_BEST_PARAMS = {
    'n_estimators'     : 1359,
    'learning_rate'    : 0.005649190003825223,
    'num_leaves'       : 249,
    'min_child_samples': 153,
    'subsample'        : 0.507054270662170,
    'colsample_bytree' : 0.004327473114,
    'reg_alpha'        : 1.25178966847,
    'reg_lambda'       : 0.00197649522,
    'min_split_gain'   : 0.439518138431467,
    'scale_pos_weight' : SCALE_POS_WEIGHT,
    'n_jobs'           : -1,
    'random_state'     : 42,
    'verbose'          : -1,
}

all_oof_lgb  = []
all_test_lgb = []

for seed in SEEDS:
    print(f'\nRunning LightGBM V2 seed={seed}...')
    params = {**LGB_BEST_PARAMS, 'random_state': seed}

    def lgb_fn(X_tr, y_tr, X_val, y_val):
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set  = [(X_val, y_val)],
            callbacks = [lgb.early_stopping(EARLY_STOPPING, verbose=False),
                         lgb.log_evaluation(-1)]
        )
        return model.predict_proba(X_val)[:, 1], model.predict_proba(Xt_v2)[:, 1]

    oof, test, auc = run_cv(lgb_fn, X_v2, y, Xt_v2, seed=seed, label=f'LGB-s{seed}')
    all_oof_lgb.append(oof)
    all_test_lgb.append(test)

oof_lgb_avg  = np.mean(all_oof_lgb,  axis=0)
test_lgb_avg = np.mean(all_test_lgb, axis=0)
auc_lgb_avg  = roc_auc_score(y, oof_lgb_avg)

oof_lgb_rank  = rank_avg(all_oof_lgb)
test_lgb_rank = rank_avg(all_test_lgb)
auc_lgb_rank  = roc_auc_score(y, oof_lgb_rank)

print(f'\n✅ Multi-seed LightGBM V2:')
print(f'   Prob avg OOF AUC: {auc_lgb_avg:.5f}')
print(f'   Rank avg OOF AUC: {auc_lgb_rank:.5f}')


Running LightGBM V2 seed=42...
  [LGB-s42] Fold 1/5 — AUC: 0.92268
  [LGB-s42] Fold 2/5 — AUC: 0.91531
  [LGB-s42] Fold 3/5 — AUC: 0.93098
  [LGB-s42] Fold 4/5 — AUC: 0.92470
  [LGB-s42] Fold 5/5 — AUC: 0.92158
  [LGB-s42] OOF AUC: 0.84183 | Mean: 0.92305 ± 0.00506

Running LightGBM V2 seed=123...
  [LGB-s123] Fold 1/5 — AUC: 0.94442
  [LGB-s123] Fold 2/5 — AUC: 0.94575
  [LGB-s123] Fold 3/5 — AUC: 0.94372
  [LGB-s123] Fold 4/5 — AUC: 0.94502
  [LGB-s123] Fold 5/5 — AUC: 0.94485
  [LGB-s123] OOF AUC: 0.94474 | Mean: 0.94475 ± 0.00067

Running LightGBM V2 seed=456...
  [LGB-s456] Fold 1/5 — AUC: 0.93901
  [LGB-s456] Fold 2/5 — AUC: 0.94072
  [LGB-s456] Fold 3/5 — AUC: 0.93915
  [LGB-s456] Fold 4/5 — AUC: 0.93813
  [LGB-s456] Fold 5/5 — AUC: 0.93719
  [LGB-s456] OOF AUC: 0.93882 | Mean: 0.93884 ± 0.00117

Running LightGBM V2 seed=789...
  [LGB-s789] Fold 1/5 — AUC: 0.94960
  [LGB-s789] Fold 2/5 — AUC: 0.94900
  [LGB-s789] Fold 3/5 — AUC: 0.94997
  [LGB-s789] Fold 4/5 — AUC: 0.94928
  [L

## 6. Multi-Seed CatBoost V1

In [6]:
# Best CatBoost V1 params (from 04_modeling.ipynb Optuna run)
CB_BEST_PARAMS = {
    'iterations'         : 681,
    'learning_rate'      : 0.04177330720,
    'depth'              : 7,
    'l2_leaf_reg'        : 7.916006090621,
    'bagging_temperature': 0.566676701,
    'random_strength'    : 0.4256,
    'class_weights'      : [1, SCALE_POS_WEIGHT],
    'eval_metric'        : 'AUC',
    'verbose'            : False,
}

all_oof_cb  = []
all_test_cb = []

for seed in SEEDS:
    print(f'\nRunning CatBoost seed={seed}...')
    params = {**CB_BEST_PARAMS, 'random_seed': seed}

    def cb_fn(X_tr, y_tr, X_val, y_val):
        model = cb.CatBoostClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set              = (X_val, y_val),
            early_stopping_rounds = EARLY_STOPPING,
            verbose               = False
        )
        return model.predict_proba(X_val)[:, 1], model.predict_proba(Xt_v1)[:, 1]

    oof, test, auc = run_cv(cb_fn, X_v1, y, Xt_v1, seed=seed, label=f'CB-s{seed}')
    all_oof_cb.append(oof)
    all_test_cb.append(test)

oof_cb_avg  = np.mean(all_oof_cb,  axis=0)
test_cb_avg = np.mean(all_test_cb, axis=0)
auc_cb_avg  = roc_auc_score(y, oof_cb_avg)

oof_cb_rank  = rank_avg(all_oof_cb)
test_cb_rank = rank_avg(all_test_cb)
auc_cb_rank  = roc_auc_score(y, oof_cb_rank)

print(f'\n✅ Multi-seed CatBoost V1:')
print(f'   Prob avg OOF AUC: {auc_cb_avg:.5f}')
print(f'   Rank avg OOF AUC: {auc_cb_rank:.5f}')


Running CatBoost seed=42...
  [CB-s42] Fold 1/5 — AUC: 0.96822
  [CB-s42] Fold 2/5 — AUC: 0.96924
  [CB-s42] Fold 3/5 — AUC: 0.96787
  [CB-s42] Fold 4/5 — AUC: 0.96903
  [CB-s42] Fold 5/5 — AUC: 0.97011
  [CB-s42] OOF AUC: 0.96885 | Mean: 0.96889 ± 0.00079

Running CatBoost seed=123...
  [CB-s123] Fold 1/5 — AUC: 0.96878
  [CB-s123] Fold 2/5 — AUC: 0.96955
  [CB-s123] Fold 3/5 — AUC: 0.96838
  [CB-s123] Fold 4/5 — AUC: 0.96927
  [CB-s123] Fold 5/5 — AUC: 0.96903
  [CB-s123] OOF AUC: 0.96896 | Mean: 0.96900 ± 0.00040

Running CatBoost seed=456...
  [CB-s456] Fold 1/5 — AUC: 0.97031
  [CB-s456] Fold 2/5 — AUC: 0.96728
  [CB-s456] Fold 3/5 — AUC: 0.97041
  [CB-s456] Fold 4/5 — AUC: 0.96884
  [CB-s456] Fold 5/5 — AUC: 0.96833
  [CB-s456] OOF AUC: 0.96898 | Mean: 0.96904 ± 0.00119

Running CatBoost seed=789...
  [CB-s789] Fold 1/5 — AUC: 0.96763
  [CB-s789] Fold 2/5 — AUC: 0.96967
  [CB-s789] Fold 3/5 — AUC: 0.96903
  [CB-s789] Fold 4/5 — AUC: 0.96980
  [CB-s789] Fold 5/5 — AUC: 0.96841
  

## 7. Final Ensemble Comparison

We compare all ensembling strategies and pick the best for submission.

In [7]:
# ── Strategy 1: Probability average of multi-seed models ──────────────────
oof_prob_avg  = (oof_xgb_avg + oof_lgb_avg + oof_cb_avg) / 3
test_prob_avg = (test_xgb_avg + test_lgb_avg + test_cb_avg) / 3
auc_prob_avg  = roc_auc_score(y, oof_prob_avg)

# ── Strategy 2: Rank average of multi-seed models ─────────────────────────
oof_rank_avg  = rank_avg([oof_xgb_avg, oof_lgb_avg, oof_cb_avg])
test_rank_avg = rank_avg([test_xgb_avg, test_lgb_avg, test_cb_avg])
auc_rank_avg  = roc_auc_score(y, oof_rank_avg)

# ── Strategy 3: Weighted by individual AUC ────────────────────────────────
aucs    = np.array([auc_xgb_avg, auc_lgb_avg, auc_cb_avg])
weights = aucs / aucs.sum()
oof_weighted  = weights[0]*oof_xgb_avg + weights[1]*oof_lgb_avg + weights[2]*oof_cb_avg
test_weighted = weights[0]*test_xgb_avg + weights[1]*test_lgb_avg + weights[2]*test_cb_avg
auc_weighted  = roc_auc_score(y, oof_weighted)

# ── Strategy 4: Stacking meta-learner ─────────────────────────────────────
oof_meta  = np.column_stack([oof_xgb_avg, oof_lgb_avg, oof_cb_avg])
test_meta = np.column_stack([test_xgb_avg, test_lgb_avg, test_cb_avg])
scaler    = StandardScaler()
meta      = LogisticRegression(C=1.0, max_iter=1000, random_state=42, class_weight='balanced')
meta.fit(scaler.fit_transform(oof_meta), y)
oof_stack  = meta.predict_proba(scaler.transform(oof_meta))[:, 1]
test_stack = meta.predict_proba(scaler.transform(test_meta))[:, 1]
auc_stack  = roc_auc_score(y, oof_stack)

# ── Strategy 5: XGBoost multi-seed alone (rank avg) ───────────────────────
auc_xgb_ms = roc_auc_score(y, oof_xgb_rank)

# ── Compare all strategies ─────────────────────────────────────────────────
strategies = {
    'XGB multi-seed rank avg (alone)': (auc_xgb_ms,   test_xgb_rank),
    'XGB multi-seed prob avg (alone)': (auc_xgb_avg,  test_xgb_avg),
    'Prob avg (XGB+LGB+CB)'          : (auc_prob_avg,  test_prob_avg),
    'Rank avg (XGB+LGB+CB)'          : (auc_rank_avg,  test_rank_avg),
    'Weighted avg (XGB+LGB+CB)'      : (auc_weighted,  test_weighted),
    'Stacking meta-learner'          : (auc_stack,     test_stack),
}

print('OOF AUC Comparison:')
print('-' * 55)
best_name, best_auc, best_preds = None, 0, None
for name, (auc, preds) in sorted(strategies.items(), key=lambda x: -x[1][0]):
    marker = ' ← BEST' if auc == max(s[0] for s in strategies.values()) else ''
    print(f'  {name:40s}: {auc:.5f}{marker}')
    if auc > best_auc:
        best_auc, best_name, best_preds = auc, name, preds

print(f'\nMeta-learner coefficients (XGB, LGB, CB): {meta.coef_[0].round(4)}')
print(f'Weights (weighted avg): XGB={weights[0]:.4f}, LGB={weights[1]:.4f}, CB={weights[2]:.4f}')

OOF AUC Comparison:
-------------------------------------------------------
  XGB multi-seed rank avg (alone)         : 0.97104 ← BEST
  XGB multi-seed prob avg (alone)         : 0.97102
  Weighted avg (XGB+LGB+CB)               : 0.97061
  Prob avg (XGB+LGB+CB)                   : 0.97061
  Stacking meta-learner                   : 0.96787
  Rank avg (XGB+LGB+CB)                   : 0.96764

Meta-learner coefficients (XGB, LGB, CB): [0.8657 0.8598 0.5449]
Weights (weighted avg): XGB=0.3363, LGB=0.3279, CB=0.3358


## 8. Save All Submissions

In [8]:
# ── Save primary (best) submission ────────────────────────────────────────
pd.DataFrame({'id': test_v1['id'], 'order_placed': best_preds}).to_csv(SUB_PATH, index=False)
print(f'✅ Primary submission saved: {SUB_PATH}')
print(f'   Method : {best_name}')
print(f'   OOF AUC: {best_auc:.5f}')

# ── Save all strategies as separate files ─────────────────────────────────
for name, (auc, preds) in strategies.items():
    fname = name.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('+', '').replace('/', '') + '.csv'
    path  = os.path.join(OUT_DIR, f'sub_{fname}')
    pd.DataFrame({'id': test_v1['id'], 'order_placed': preds}).to_csv(path, index=False)
    print(f'   Saved: sub_{fname}  (OOF AUC: {auc:.5f})')

✅ Primary submission saved: c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\outputs\submission_final.csv
   Method : XGB multi-seed rank avg (alone)
   OOF AUC: 0.97104
   Saved: sub_xgb_multi-seed_rank_avg_alone.csv  (OOF AUC: 0.97104)
   Saved: sub_xgb_multi-seed_prob_avg_alone.csv  (OOF AUC: 0.97102)
   Saved: sub_prob_avg_xgblgbcb.csv  (OOF AUC: 0.97061)
   Saved: sub_rank_avg_xgblgbcb.csv  (OOF AUC: 0.96764)
   Saved: sub_weighted_avg_xgblgbcb.csv  (OOF AUC: 0.97061)
   Saved: sub_stacking_meta-learner.csv  (OOF AUC: 0.96787)


## 9. Results Summary

In [9]:
print('=' * 60)
print('FINAL ENSEMBLE RESULTS SUMMARY')
print('=' * 60)
print(f'  XGBoost  : 5-seed avg, V1 features (29), best params')
print(f'  LightGBM : 5-seed avg, V2 features (52), best params')
print(f'  CatBoost : 5-seed avg, V1 features (29), best params')
print('-' * 60)
for name, (auc, _) in sorted(strategies.items(), key=lambda x: -x[1][0]):
    marker = ' ← SUBMITTED' if name == best_name else ''
    print(f'  {name:40s}: {auc:.5f}{marker}')
print('=' * 60)
print(f'\n  Previous best (public LB) : 0.97031')
print(f'  Leaderboard top           : 0.97188')
print(f'  Gap to close              : {0.97188 - best_auc:.5f} (CV estimate)')

FINAL ENSEMBLE RESULTS SUMMARY
  XGBoost  : 5-seed avg, V1 features (29), best params
  LightGBM : 5-seed avg, V2 features (52), best params
  CatBoost : 5-seed avg, V1 features (29), best params
------------------------------------------------------------
  XGB multi-seed rank avg (alone)         : 0.97104 ← SUBMITTED
  XGB multi-seed prob avg (alone)         : 0.97102
  Weighted avg (XGB+LGB+CB)               : 0.97061
  Prob avg (XGB+LGB+CB)                   : 0.97061
  Stacking meta-learner                   : 0.96787
  Rank avg (XGB+LGB+CB)                   : 0.96764

  Previous best (public LB) : 0.97031
  Leaderboard top           : 0.97188
  Gap to close              : 0.00084 (CV estimate)
